
# Diabetes Prediction Using Machine Learning

## Project Overview
This project aims to predict diabetes using patient demographic and medical information.

### Workflow
1. Data Loading
2. Data Inspection
3. Data Cleaning
4. Exploratory Data Analysis
5. Feature Engineering
6. Model Training
7. Hyperparameter Tuning
8. Threshold Optimization
9. Feature Importance Analysis
10. Model Comparison
11. Final Model Selection
12. Deployment Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#  Imports

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)

from sklearn.inspection import permutation_importance

import joblib

# 📂 Data Loading

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/diabetes_prediction_dataset.csv.zip")

# 🔍 Data Inspection

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.nunique()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

# 🧹 Data Cleaning

In [ ]:
df = df.drop_duplicates()

### Separating Numeric and Categorical Columns

In [ ]:
numeric_cols = df.select_dtypes(include=['number'])
categorical_cols = df.select_dtypes(include=['object', 'category'])

print("Numeric Columns:")
display(numeric_cols.head())

print("\nCategorical Columns:")
display(categorical_cols.head())

In [ ]:

"""
NOTE!
Blood glucose levels can range
From 70 to 600 ml/dl, so the
Outlier is stastical
not in medical terms.

And hba1c can more then 9
So again the outlier is stastical
And it is possible in many patients
To have 11 to 14 hba1c level.
"""
for col in numeric_cols.columns:
    sns.boxplot(y = numeric_cols[col])
    plt.show()

print("Rows with BMI > 70:", (df['bmi'] > 70).sum())
df = df[df["age"]>1]

df =df[(df["bmi"]< 65) & (df["bmi"]> 12) ]

# 📊 Exploratory Data Analysis

In [ ]:
for col in numeric_cols.columns:
    sns.boxplot(y = numeric_cols[col])
    plt.show()

In [ ]:
for col in numeric_cols.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[col], kde=True)
    plt.title(col)
    plt.show()

df = df[df['gender'].isin(['Male', 'Female'])]
print(df ["gender"].nunique())
print(df ["gender"].value_counts())

In [ ]:
for col in categorical_cols.columns:
    plt.figure(figsize=(8,4))
    sns.countplot(df[col])
    plt.title(col)
    plt.show()

In [ ]:
plt.figure(figsize= (10 , 8))
sns.heatmap(
    df.corr(numeric_only = True) ,
    annot= True ,
    cmap = "coolwarm"
)
plt.title("relationship between features")
plt.show()

In [ ]:
sns.boxplot(x='diabetes', y='blood_glucose_level', data=df)
plt.show()

sns.boxplot(x='diabetes', y='HbA1c_level', data=df)
plt.show()

sns.boxplot(x='diabetes', y='bmi', data=df)
plt.show()

sns.boxplot(x="diabetes", y ="age",data = df)
plt.show()

In [ ]:
df.columns
df['smoking_history'].unique()

sns.countplot(x='smoking_history', hue='diabetes', data=df)
plt.xticks(rotation=45)
plt.show()

#EDA Conclusion

After exploring the dataset, several important patterns were identified. The data contained both numerical and categorical features related to diabetes risk, including age, BMI, HbA1c level, blood glucose level, gender, smoking history, hypertension, and heart disease.

During data cleaning, records with BMI values greater than 70 were removed as they were considered extremely uncommon and could potentially affect the analysis. Records with gender labeled as "Other" were also removed because they represented only a very small portion of the dataset. All age groups were retained, excluding individuals younger than one year old, to preserve the full range of ages available in the data.

The target variable, diabetes, was found to be imbalanced, with non-diabetic cases occurring much more frequently than diabetic cases. This will need to be considered during model training.

The visualizations provided useful insights into the relationship between different features and diabetes. Individuals with diabetes generally had higher blood glucose levels, higher HbA1c levels, higher BMI values, and were older compared to individuals without diabetes. Among all features, HbA1c level and blood glucose level showed the strongest relationship with diabetes, while BMI and age also appeared to be important factors.

The correlation analysis supported these observations, showing that HbA1c level, blood glucose level, BMI, and age have the strongest association with the target variable.

Overall, the exploratory analysis helped identify the most relevant features and provided a better understanding of the dataset. The data is now ready for preprocessing and the development

# ⚙️ Feature Engineering

# encoding
What encoding means , making input value x numeric so a machine learning model can understand it , why it is important because machine learning models can't read character they can only understand numbers.

# separating the target variable and encoding the categorical columns.

In [ ]:
x = df.drop("diabetes" ,axis = 1 )
y = df["diabetes"]

x = pd.get_dummies(
    x ,
    columns=['gender', 'smoking_history'],
    drop_first = True
 )

for col in x.select_dtypes(include='bool').columns:
    x[col] = x[col].astype(int)
x

# ✂️ Train Test Split

# trian_test__split
It's important to split your data before you do standardscaler cause the there might be chances of of the data getting leaked

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numerical_features = [
    "age",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

scaler = StandardScaler()

x_train[numerical_features] = scaler.fit_transform(
    x_train[numerical_features]
)

x_test[numerical_features] = scaler.transform(
    x_test[numerical_features]
)

#  Model Training

In [ ]:

model = LogisticRegression(
    class_weight="balanced",
    random_state=42)

model.fit(x_train, y_train)


pred = model.predict(x_test)

In [ ]:
print (classification_report(y_test ,pred ))

## Model Initialization

I initialized four classification models to compare their performance on the dataset:

- Logistic Regression (baseline linear model)
- Random Forest (bagging ensemble)
- Gradient Boosting (boosting ensemble)
- XGBoost (optimized gradient boosting implementation)

All models are trained using the same training dataset to ensure a fair comparison.

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(
        random_state=42,
        max_iter=1000
    ),
    "RandomForest": RandomForestClassifier(
        random_state=42
    ),
    "GradientBoosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier()
}

## Model Training and Evaluation (Classification Report)

Each model was trained on the training dataset and evaluated on the test set.

I used classification metrics including Precision, Recall, F1-score, and Accuracy to compare performance across models.

Since the dataset is imbalanced, special attention is given to the performance on class 1 (positive class).

In [ ]:
for name, model in models.items():
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    print(f" {name}")
    print(classification_report(y_test, pred))

## Logistic Regression Hyperparameter Tuning

To improve model performance, Grid Search Cross Validation was used to find the best combination of hyperparameters for Logistic Regression.

The search explored different values of:

- C (regularization strength)
- Solver
- Class weights

The F1-score was used as the optimization metric because the dataset is imbalanced and F1-score provides a balance between precision and recall.

After identifying the best hyperparameters, the optimized model was evaluated on the test dataset.

In [ ]:

lr_param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['liblinear' , "saga"],
    'class_weight': [None, 'balanced']

}


grid = GridSearchCV(
    estimator = models["LogisticRegression"],
    param_grid = lr_param_grid ,
    cv = 3 ,
    n_jobs = -1,
    scoring= "f1"

)

grid.fit(x_train , y_train)

best_lr = grid.best_estimator_
best_params = grid.best_params_

y_pred = best_lr.predict(x_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix)

## Random Forest Hyperparameter Tuning

Grid Search Cross Validation was used to identify the best hyperparameters for the Random Forest classifier.

The search focused on:

- Number of trees (`n_estimators`)
- Maximum tree depth (`max_depth`)
- Minimum samples required to split a node (`min_samples_split`)
- Minimum samples required at a leaf node (`min_samples_leaf`)
- Class weighting (`class_weight`)

The F1-score was used as the optimization metric to balance precision and recall on the imbalanced dataset.

### Best Parameters

- n_estimators: 300
- max_depth: None
- min_samples_split: 5
- min_samples_leaf: 2
- class_weight: balanced.

In [ ]:

rf_param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None ,100],
    'min_samples_split': [ 5],
    'min_samples_leaf': [ 2],
    'class_weight': [ 'balanced']
}


grid = GridSearchCV(
    estimator = models["RandomForest"],
    param_grid = rf_param_grid ,
    cv = 3 ,
    n_jobs = -1,
    scoring= "f1"

)

grid.fit(x_train , y_train)

best_rf = grid.best_estimator_
best_params = grid.best_params_

In [ ]:
print(best_params)
y_pred2 = best_rf.predict(x_test)
print(classification_report(y_test , y_pred2))
probs = best_rf.predict_proba(x_test)[:, 1]

for threshold in [0.1 , 0.2 , 0.3, 0.4, 0.5, 0.67, 0.7]:
    preds = (probs >= threshold).astype(int)

    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)

    print(
        f"Threshold={threshold} "
        f"Precision={precision:.3f} "
        f"Recall={recall:.3f}"
    )
    print (confusion_matrix(y_test , preds))

## XGBoost Hyperparameter Tuning

Grid Search Cross Validation was performed to optimize the XGBoost classifier.

The search explored combinations of:

- Number of trees (`n_estimators`)
- Tree depth (`max_depth`)
- Learning rate (`learning_rate`)
- Row sampling ratio (`subsample`)
- Feature sampling ratio (`colsample_bytree`)

The F1-score was used as the optimization metric to balance precision and recall on the imbalanced dataset.

## Tuned XGBoost Results

Metric      Class 1
Precision   0.97
Recall      0.68
F1-Score    0.80

Overall Accuracy: 97%

The tuned XGBoost model achieved the highest precision among all evaluated models while maintaining a strong F1-score. This indicates that when the model predicts a positive case, it is usually correct.
## Best Parameters

 Parameter     |    Value

 n_estimators       300
 max_depth          3
 learning_rate      0.1
subsample           1.0
colsample_bytree    1.0

In [ ]:
xg_grid = {
    'n_estimators': [300, 200],
    'max_depth': [3 , 5 , 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]


}

grid = GridSearchCV(
    estimator = models["XGBoost"],
    param_grid = xg_grid,
    cv = 3,
    n_jobs = -1,
    scoring = "f1"
)

grid.fit(x_train , y_train)
best_xg = grid.best_estimator_
best_params = grid.best_params_

In [ ]:
print(best_params)
y_pred2 = best_xg.predict(x_test)
print(classification_report(y_test , y_pred2))


probs = best_xg.predict_proba(x_test)[:, 1]

for threshold in [0.17 , 0.27 , 0.3, 0.32, 0.5, 0.67, 0.7]:
    preds = (probs >= threshold).astype(int)

    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)

    print(
        f"Threshold={threshold} "
        f"Precision={precision:.3f} "
        f"Recall={recall:.3f}"
    )
    print (confusion_matrix(y_test , preds))

## Interpretation

Compared to the other models, XGBoost achieved the strongest balance between precision and recall after threshold optimization.

The model was selected as the final model because:

- It achieved the highest F1-score after threshold tuning.
- It provided a better balance between detecting positive cases and limiting false positives.

In [ ]:



gb_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],

}

grid = GridSearchCV(
    estimator=models["GradientBoosting"],
    scoring="f1",
    param_grid=gb_grid,
    cv=3,
    n_jobs=-1
)

grid.fit(x_train ,y_train)
best_gb = grid.best_estimator_
best_params = grid.best_params_

# Threshold Optimization

## Gradient Boosting Interpretation

The tuned Gradient Boosting model achieved strong overall performance, with an accuracy of 97% and an F1-score of 0.80 for the positive class.

The model demonstrated very high precision (0.97), indicating that most positive predictions were correct. However, the recall of 0.68 shows that some positive cases were still missed when using the default threshold of 0.5.

Threshold tuning revealed the expected trade-off between precision and recall:

- Lower thresholds increased recall but produced more false positives.
- Higher thresholds increased precision but reduced recall.

A threshold of 0.20 provided the best balance between precision (0.698) and recall (0.814), while a threshold of 0.30 offered a more conservative balance with precision (0.844) and recall (0.741).

Overall, Gradient Boosting performed competitively with XGBoost and achieved one of the highest F1-scores among the evaluated models. Its strong balance between precision and recall makes it a reliable choice for diabetes prediction.

In [ ]:
print(best_params)
y_pred3 = best_gb.predict(x_test)
print(classification_report(y_test , y_pred3))


probs = best_gb.predict_proba(x_test)[:, 1]

for threshold in [0.1 , 0.2 , 0.3, 0.4, 0.5, 0.67, 0.7]:
    preds = (probs >= threshold).astype(int)

    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)

    print(
        f"Threshold={threshold} "
        f"Precision={precision:.3f} "
        f"Recall={recall:.3f}"
    )
    print (confusion_matrix(y_test , preds))

## Feature Importance Analysis

Permutation importance was used to measure the contribution of each feature to the performance of the final XGBoost model.

The results show that the model relies heavily on two features:

- HbA1c_level (0.401)
- blood_glucose_level (0.327)

These features had substantially higher importance scores than all other variables, indicating that they are the strongest predictors of diabetes in this dataset.

Features such as BMI and age provided a smaller but still positive contribution to model performance. Hypertension and heart disease had only a minor impact on predictions.

The smoking history and gender variables showed near-zero importance scores, suggesting that they contributed very little to the model's predictive power.

### Key Findings

1. HbA1c level was the most influential feature.
2. Blood glucose level was the second most important feature.
3. BMI and age had moderate predictive value.
4. Smoking history and gender had minimal influence on model predictions.

Overall, the model's predictions were driven primarily by clinical measurements rather than demographic or lifestyle-related features.

In [ ]:

result = permutation_importance(
    best_xg,
    x_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="f1"
)

importance = pd.DataFrame({
    "Feature": x_test.columns,
    "Importance": result.importances_mean
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

## Final Model Comparison

The models were compared using optimized decision thresholds and evaluated based on Precision, Recall, and F1-score.

| Model | Threshold | Precision | Recall | F1 Score |
|---------|---------|---------|---------|---------|
| XGBoost | 0.27 | 0.805 | 0.763 | 0.784 |
| Gradient Boosting | 0.20 | 0.698 | 0.814 | 0.752 |
| Random Forest | 0.40 | 0.707 | 0.775 | 0.739 |
| Logistic Regression | 0.50 | 0.851 | 0.633 | 0.726 |

### Interpretation

- XGBoost achieved the highest F1-score, indicating the best overall balance between precision and recall.
- Gradient Boosting achieved the highest recall, making it effective at identifying positive cases, but at the cost of lower precision.
- Random Forest provided a balanced performance but did not outperform the boosting-based models.
- Logistic Regression achieved the highest precision among the baseline models but had the lowest recall, meaning it missed more positive cases.

### Final Model Selection

XGBoost was selected as the final model because it achieved the strongest balance between precision and recall, resulting in the highest F1-score among all evaluated models. This makes it the most reliable model for diabetes prediction in this project.

In [ ]:
results = []

models = {
    "LogisticRegression": (best_lr, 0.5),
    "RandomForest": (best_rf, 0.4),
    "GradientBoosting": (best_gb, 0.2),
    "XGBoost": (best_xg, 0.27)
}

for name, (model, threshold) in models.items():

    probs = model.predict_proba(x_test)[:, 1]

    y_pred = (probs >= threshold).astype(int)

    results.append({
        "Model": name,
        "Threshold": threshold,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results)

results_df.sort_values("F1", ascending=False)

## ROC AUC Evaluation

To evaluate the ability of each model to distinguish between positive and negative cases across all possible classification thresholds, the ROC AUC score was calculated.

| Model | ROC AUC |
|---------|---------|
| Logistic Regression | 0.9616 |
| Random Forest | 0.9733 |
| Gradient Boosting | 0.9783 |
| XGBoost | 0.9780 |

### Interpretation

- All models achieved excellent ROC AUC scores (> 0.96), indicating strong discriminative performance.
- Gradient Boosting achieved the highest ROC AUC score (0.9783), closely followed by XGBoost (0.9780).
- Logistic Regression performed well but was outperformed by the ensemble-based models.
- The small difference between Gradient Boosting and XGBoost suggests that both models are highly effective for this classification task.

### ROC Curve

The ROC curve for the final XGBoost model was plotted to visualize the trade-off between the True Positive Rate (Recall) and False Positive Rate across different classification thresholds.

A curve closer to the upper-left corner indicates stronger classification performance. The high ROC AUC score confirms that the model can effectively separate diabetic and non-diabetic cases.

In [ ]:
for name, (model, threshold) in models.items():

    probs = model.predict_proba(x_test)[:,1]

    auc = roc_auc_score(y_test, probs)

    print(name, auc)


RocCurveDisplay.from_estimator(
    best_xg,
    x_test,
    y_test
)


# Final Model Selection

### Selected Model
**XGBoost**

### Final Threshold
**0.27**

### Key Metrics
- ROC-AUC:97  
- Precision:80
- Recall:76
- F1 Score:78

### Key Finding
HbA1c level and blood glucose level were the most influential predictors.


# Deployment Preparation

In [ ]:
"""
BEST_THRESHOLD = 0.27

joblib.dump(best_xg, 'diabetes_model.pkl')
joblib.dump(list(x_train.columns), 'feature_names.pkl')
joblib.dump(BEST_THRESHOLD, 'threshold.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('Saved: diabetes_model.pkl')
print('Saved: feature_names.pkl')
print('Saved: threshold.pkl')
print('Saved: scaler.pkl')
"""

In [ ]:
import os

files = [
    "DIABETES_PROJECT.ipynb",
    "diabetes_model.pkl",
    "scaler.pkl",
    "feature_names.pkl",
    "threshold.pkl"
]

for file in files:
    if os.path.exists(file):
        print(file, round(os.path.getsize(file)/1024/1024,2), "MB")